In [3]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

import os

load_dotenv()

llm = ChatOpenAI(
    base_url = os.getenv("OPENAI_BASE_URL"),
    api_key = os.getenv("OPENAI_API_KEY"), 
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
    temperature = 0.1,
)

In [10]:
from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a Python programming tutor. Explain concepts clearly, simply, and with examples when helpful."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

def load_memory(_):
    return memory.load_memory_variables({})["history"]

chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )
    print(result)

invoke_chain("What is a variable?")
invoke_chain("What is a function?")
invoke_chain("What is the difference between a for loop and a while loop in Python?")
print("Memory status: ", llm.get_num_tokens_from_messages(memory.load_memory_variables({})["history"]), "tokens")

content='A variable is a named “box” in memory that stores a value so you can use it later in your program.\n\nIn Python:\n\n```python\nage = 15\nname = "Sam"\npi = 3.14\n```\n\n- `age`, `name`, and `pi` are variables.\n- `=` is the assignment operator (it means “store this value in this name”).\n- The values are `15`, `"Sam"`, and `3.14`.\n\nYou can then use the variable instead of the literal value:\n\n```python\nprint(age)      # prints 15\nage = age + 1   # increase age by 1\nprint(age)      # prints 16\n```\n\nKey points:\n- A variable has a **name** (like `age`),\n- holds a **value** (like `15`),\n- and that value can **change** as the program runs.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 200, 'prompt_tokens': 34, 'total_tokens': 234, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_toke